# 🏋️ Exercise 1 — Extract
**Tujuan:** mengambil *seluruh* data penjualan dari API yang menggunakan pagination.

> Isi bagian bertanda `# TODO`, lalu jalankan sel **Cek Jawaban**.

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [6]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

Konfigurasi OK — login database sebagai peserta25 @ ep-wandering-math-b3bml8w1-pooler.c-4.ap-southeast-1.aws.neon.tech


In [7]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

## Tugas
1. Lengkapi fungsi `extract_sales()` agar mengambil semua halaman (ulang selama `has_next` bernilai `True`).
2. Simpan hasilnya ke variabel `df_raw`.

Petunjuk: `requests.get(page_url(api_url, page), timeout=30)`, lalu `resp.json()["data"]`. Fungsi `page_url` sudah tersedia di sel import.

In [18]:
def extract_sales(api_url, page_size=100):
    rows, page = [], 1

    while True:
        # TODO 1: Kirim request ke URL halaman
        resp = requests.get(
            page_url(api_url, page, page_size),
            timeout=30
        )

        # TODO 2: Cek status dan ambil JSON
        resp.raise_for_status()
        payload = resp.json()

        # TODO 3: Tambahkan data ke rows
        rows.extend(payload["data"])

        # TODO 4: Cek apakah masih ada halaman berikutnya
        if not payload.get("has_next", False):
            break

        page += 1

    return pd.DataFrame(rows)


# Simpan hasil ekstraksi ke DataFrame
df_raw = extract_sales(API_URL)

# Tampilkan 5 baris pertama
df_raw.head()

,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price
0,TRX00001,2026-01-15,C040,SURABAYA,P07,Upgrade Bandwidth,Layanan,1,200000
1,TRX00002,2026-01-28,C057,Malang,P03,Paket Internet 100 Mbps,Internet,1,650000
2,TRX00003,2026-02-19,C060,batu,P03,Paket Internet 100 Mbps,Internet,1,650000
3,TRX00004,2026-02-05,C052,surabaya,P05,Access Point Mesh,Perangkat,1,850000
4,TRX00005,2026-03-23,C028,Pasuruan,P03,Paket Internet 100 Mbps,Internet,1,650000


In [ ]:
df_raw.head(304)

,transaction_id,transaction_date,customer_id,city,product_id,product_name,category,quantity,unit_price
0,TRX00001,2026-01-15,C040,SURABAYA,P07,Upgrade Bandwidth,Layanan,1,200000
1,TRX00002,2026-01-28,C057,Malang,P03,Paket Internet 100 Mbps,Internet,1,650000
2,TRX00003,2026-02-19,C060,batu,P03,Paket Internet 100 Mbps,Internet,1,650000
3,TRX00004,2026-02-05,C052,surabaya,P05,Access Point Mesh,Perangkat,1,850000
4,TRX00005,2026-03-23,C028,Pasuruan,P03,Paket Internet 100 Mbps,Internet,1,650000
...,...,...,...,...,...,...,...,...,...
299,TRX00296,2026-01-14,C027,sidoarjo,P06,Biaya Instalasi,Layanan,2,-150000
300,TRX00297,04/02/2026,C016,SURABAYA,P07,Upgrade Bandwidth,Layanan,1,200000
301,TRX00298,2026-01-06,C033,malang,P01,Paket Internet 20 Mbps,Internet,3,250000
302,TRX00299,2026-02-04,C041,Surabaya,P01,Paket Internet 20 Mbps,Internet,,250000


## Cek Jawaban

In [16]:
assert isinstance(df_raw, pd.DataFrame), "df_raw harus DataFrame"
assert len(df_raw) == 304, f"Jumlah baris {len(df_raw)}, seharusnya 304 — sudah mengambil semua halaman?"
assert list(df_raw.columns)[:3] == ["transaction_id", "transaction_date", "customer_id"]
print("✅ Exercise 1 benar! Anda berhasil mengambil", len(df_raw), "baris.")

✅ Exercise 1 benar! Anda berhasil mengambil 304 baris.


## Tantangan tambahan
1. Tambahkan *retry* (coba ulang maksimal 3 kali) jika request gagal.
2. Simpan data mentah ke file `sales_raw_backup.csv` — praktik umum menyimpan *raw layer*.

In [ ]:
# tantangan